# Flausch -- Subtask 2: first span detector then span classifier

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [4]:
import pandas as pd
import os
import sys
import numpy as np

# work with the train data only /content/drive/MyDrive/FlauschData/train_task1.json
#path = "/content/drive/MyDrive/FlauschData/"
path = "Input/Data/train/"
data = pd.read_json(path + "train_task1.json")
dataflausch = data[['document', 'comment_id', 'comment', 'flausch',  'spans', 'span_pairs',
       'types', 'id']].copy()
data2 = pd.read_json(path + "train_task2.json")
data2 = data2[['document', 'comment_id', 'type', 'start', 'end', 'comment', 'flausch', 'span', 'id']].copy()


## Step 1: Train span detector

In [5]:
# generate BIO tag scheme (begin/inside/outside)
# dictionaries to map between tag ids and tag names
tags = sorted(data2["type"].unique())
tag2id = {'O': 0, 'B': 1, 'I': 2, 'E': 3}
id2tag = {v: k for k, v in tag2id.items()}


In [6]:
# set checkpoint and load tokenizer
from transformers import AutoTokenizer
checkpoint = "distilbert/distilbert-base-german-cased"
model_name = checkpoint.split("/")[-1] +"_non_labeled_spans" 
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


In [7]:
#some testing
tokenized = tokenizer(["Das ist ein Horrorfilm!"], return_offsets_mapping=True)
print(tokenized.encodings[0])
print(tokenized.encodings[0].tokens)
print(tokenized.encodings[0].ids)
print(tokenized.encodings[0].offsets)


Encoding(num_tokens=8, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])
['[CLS]', 'Das', 'ist', 'ein', 'Horror', '##film', '!', '[SEP]']
[102, 347, 215, 143, 28816, 5241, 3330, 103]
[(0, 0), (0, 3), (4, 7), (8, 11), (12, 18), (18, 22), (22, 23), (0, 0)]


In [8]:
# main function that creates target token sequences for tokenized training data

def align_labels_with_tokens(tokenizer, data, tag2id):
    # tokenize text and get offset_mapping
    text = data["comment"]
    span_pairs = data["span_pairs"]  # list of boundaries of labeled flausch spans
    tokenized_input = tokenizer(text, return_offsets_mapping=True, truncation=True, max_length=512)
    # encodings[0] für den einzelnen Text
    encoding = tokenized_input.encodings[0]

    token_ids = encoding.ids  # Token-IDs
    label_ids = [tag2id['O'] for i in  range(len(encoding.tokens))]  # Initialisiere mit O-Labels für jedes Wort/Token


    # Iteriere über jeden Span, der gelabelt werden soll
    for i in range(len(span_pairs)):
        span_start_char = span_pairs[i][0]
        span_end_char = span_pairs[i][1]

        # Finde die Token-Indizes, die diesen Span abdecken
        token_start_idx = encoding.char_to_token(span_start_char)
        token_end_idx = encoding.char_to_token(span_end_char - 1) # end_char ist exklusiv

        # Wenn der Span nicht durch die Tokenisierung abgedeckt wird (z.B. wegen Truncation)
        if token_start_idx is None or token_end_idx is None:
            continue

        # Wenn der Span über mehrere Tokens geht oder ein einzelnes Token ist
        for current_token_idx in range(token_start_idx, token_end_idx + 1):
            if current_token_idx == token_start_idx:
                # Dies ist das erste Token, das den Span abdeckt -> B-Tag
                label_ids[current_token_idx] = tag2id["B"]
            elif current_token_idx == token_end_idx:
                # Dies ist das letzte Token, das den Span abdeckt -> E-Tag
                label_ids[current_token_idx] = tag2id["E"]
            else:
                # Alle anderen Tokens, die denselben Span abdecken -> I-Tag
                label_ids[current_token_idx] = tag2id["I"]

    return tokenized_input["input_ids"], label_ids



In [9]:
i=31
input_ids, label_ids = align_labels_with_tokens(tokenizer, dataflausch.loc[i], tag2id)
ex_spans = dataflausch["spans"][i]

print(ex_spans)
for idx,j in enumerate(label_ids):
    if j != 0:
        print(id2tag[j], tokenizer.decode(input_ids[idx]))

['ich finds gar nicht so schlecht :)']
B ich
I find
I ##s
I gar
I nicht
I so
I schlecht
I :
E )


In [10]:
import pandas as pd
from datasets import Dataset, Features, Value, ClassLabel, Sequence
from sklearn.model_selection import train_test_split # Für den Split in Train/Test



all_input_ids = []
all_labels = []

print("Beginne mit der Vorverarbeitung des DataFrames...")
for idx, row in dataflausch.iterrows():
    input_ids_for_text, labels_for_text = align_labels_with_tokens(tokenizer, row, tag2id)

    # Sammle die Ergebnisse
    all_input_ids.append(input_ids_for_text)
    all_labels.append(labels_for_text)

    if (idx + 1) % 5000 == 0: # Fortschrittsanzeige alle XX Texte
        print(f"Verarbeitete {idx + 1}/{len(dataflausch)} Texte.")

print("Alle Texte erfolgreich vorverarbeitet.")
print(f"Anzahl vorbereiteter Beispiele: {len(all_input_ids)}")

max_label_id = max(id2tag.keys())
id2tag_ordered_list = [None] * (max_label_id + 1) # Erstelle eine Liste der richtigen Größe
for k, v in id2tag.items():
    id2tag_ordered_list[k] = v

features = Features({
    'input_ids': Sequence(Value('int32')), # Sequenz von Ganzzahlen für Token-IDs
    'labels': Sequence(ClassLabel(names=id2tag_ordered_list)) # Sequenz von ClassLabels für NER-Tags
})

# Erstelle ein Dictionary, das die Daten enthält
dataset_dict = {
    'input_ids': all_input_ids,
    'labels': all_labels
}

# Erstelle ein Dataset aus dem Dictionary und den definierten Features
prepared_dataset = Dataset.from_dict(dataset_dict, features=features)

print("\nHugging Face Dataset erfolgreich erstellt.")
print(prepared_dataset)
print("\nBeispiel des ersten Eintrags im Dataset:")
print(prepared_dataset[0])

# Überprüfe den ersten Eintrag noch einmal visuell
first_text = dataflausch["comment"].iloc[0] # .iloc[0] für den ersten Eintrag im DF
first_input_ids = prepared_dataset[0]['input_ids']
first_labels_ids = prepared_dataset[0]['labels']

print(f"\nOriginaltext (erster Eintrag): '{first_text}'")
print(f"Token (decoded): {[tokenizer.decode(t) for t in first_input_ids]}")
print(f"Labels (decoded): {[id2tag[l] for l in first_labels_ids]}")


# Teile das Dataset in Trainings- und Devsets auf (85% Training, 15% Development)
# Remember the current dataset is 90% of the original one. 10% test data have been already set aside
split_dataset = prepared_dataset.train_test_split(test_size=0.15, seed=42)
train_dataset = split_dataset['train']
dev_dataset = split_dataset['test']

# create one datasetdict for train, dev and test
from datasets import DatasetDict
dataset_dict = DatasetDict({
    'train': train_dataset,
    'dev': dev_dataset,
})

print(f"\nTrainingsdatensatzgröße: {len(train_dataset)}")
print(f"Entwicklungsdatensatzgröße: {len(dev_dataset)}")



Beginne mit der Vorverarbeitung des DataFrames...
Verarbeitete 20000/33351 Texte.
Verarbeitete 5000/33351 Texte.
Verarbeitete 25000/33351 Texte.
Verarbeitete 10000/33351 Texte.
Verarbeitete 15000/33351 Texte.
Verarbeitete 35000/33351 Texte.
Alle Texte erfolgreich vorverarbeitet.
Anzahl vorbereiteter Beispiele: 33351

Hugging Face Dataset erfolgreich erstellt.
Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 33351
})

Beispiel des ersten Eintrags im Dataset:
{'input_ids': [102, 1618, 383, 313, 4213, 684, 1717, 5509, 307, 255, 11505, 2827, 103], 'labels': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}

Originaltext (erster Eintrag): 'würde ich auch gerne wenn meine Freunde sie nicht schauen würden'
Token (decoded): ['[CLS]', 'würde', 'ich', 'auch', 'gerne', 'wenn', 'meine', 'Freunde', 'sie', 'nicht', 'schauen', 'würden', '[SEP]']
Labels (decoded): ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

Trainingsdatensatzgröße: 28348
Entwicklungsdatensatzgröße: 5003


In [11]:
# check some examples
i=31
print(dataflausch["comment"].iloc[i])
print(prepared_dataset[i]['input_ids'])
print(prepared_dataset[i]['labels'])
print([id2tag[prepared_dataset[i]['labels'][j]] for j in range(len(prepared_dataset[i]['labels']))])



Ich finde dich echt Sympatisch ;) ... Bist echt mutig als Junge solche Videos zu machen :) ... Viel Erfolg ... lG Alexandra
[102, 395, 8385, 1199, 8891, 5245, 9553, 191, 3464, 2530, 566, 566, 566, 5572, 8891, 11642, 214, 276, 6436, 3760, 11441, 205, 1327, 853, 2530, 566, 566, 566, 2998, 2993, 566, 566, 566, 228, 30917, 27596, 103]
[0, 1, 2, 2, 2, 2, 2, 2, 2, 3, 0, 0, 0, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 0, 0, 0, 1, 3, 0, 0, 0, 0, 0, 0, 0]
['O', 'B', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'E', 'O', 'O', 'O', 'B', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'E', 'O', 'O', 'O', 'B', 'E', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


In [12]:
!pip install seqeval
!pip install evaluate

In [13]:
import evaluate
seqeval = evaluate.load("seqeval")

In [15]:
import numpy as np
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report


tags = dataset_dict["train"].features['labels'].feature.names

tag2id = {name: i for i, name in enumerate(tags)}
id2tag = {i: name for i, name in enumerate(tags)}

label_list = list(tag2id.keys())

from seqeval.metrics import f1_score, precision_score, recall_score, classification_report

# Annahme: Diese Variablen sind bereits GLOBAL in deinem Skript definiert:
# tags = dataset_dict["train"].features['labels'].feature.names
# tag2id = {name: i for i, name in enumerate(tags)}
# id2tag = {i: name for i, name in enumerate(tags)}
# label_list = list(tag2id.keys()) # Dies ist die Liste aller deiner String-Labels (z.B. ['O', 'B-affection declaration', ...])

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Konvertierung der numerischen IDs zurück zu den Label-Strings
    # und Entfernen von Padding-Tokens (-100)
    true_labels = []
    for label_sequence in labels:
        true_labels.append([id2tag[l] for l in label_sequence if l != -100])

    true_predictions = []
    for prediction_sequence, label_sequence in zip(predictions, labels):
        true_predictions.append([
            id2tag[p] for p, l in zip(prediction_sequence, label_sequence) if l != -100
        ])

    # Berechne Metriken mit explizitem zero_division=0
    # Dies ist der Standard von seqeval und sorgt dafür, dass undefinierte Werte 0.0 werden.
    # Es hilft, die UndefinedMetricWarning zu kontrollieren.
    precision = precision_score(true_labels, true_predictions, zero_division=0)
    recall = recall_score(true_labels, true_predictions, zero_division=0)
    f1 = f1_score(true_labels, true_predictions, zero_division=0)

    # Generiere und gib den detaillierten Klassifikationsbericht aus.
    # Das `labels=label_list` Argument stellt sicher, dass alle deine definierten Labels
    # im Bericht erscheinen, auch wenn sie im aktuellen Batch nicht vorkommen.
    report = classification_report(
        true_labels,
        true_predictions,
        digits=4, # Anzahl der Dezimalstellen im Bericht
        zero_division=0, # Steuerung der 0-Division für den Bericht
        #labels=label_list # Wichtig, um alle Labels im Bericht zu haben führt aber zu Versionsproblemen
    )

    print("\n--- Classification Report (Aktueller Evaluationsschritt) ---")
    print(report)
    print("-----------------------------------------------------------")

    return {"precision": precision, "recall": recall, "f1": f1}




In [16]:
from transformers import DataCollatorForTokenClassification
data_collator = DataCollatorForTokenClassification(tokenizer)

In [17]:
# load model with the correct head for token classification with the number of labels
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

num_labels = len(tag2id)  # Anzahl der Labels

model = AutoModelForTokenClassification.from_pretrained(
    checkpoint, num_labels=num_labels, id2label=id2tag, label2id=tag2id
)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/270M [00:00<?, ?B/s]

c:\Users\Wiebke Petersen\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Wiebke Petersen\.cache\huggingface\hub\models--distilbert--distilbert-base-german-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Some weights of DistilBertForTokenClassification were not initialized from the model

In [ ]:
# huggingface login
from huggingface_hub import login
login()


In [ ]:
from transformers import TrainingArguments, Trainer
import os



# Define training parameters
training_args = TrainingArguments(
    output_dir="flausch_span_"+ model_name, # Verzeichnis für Modell, Checkpoints und Logs
    learning_rate=2e-5,
    metric_for_best_model="eval_f1", # competition ranking metric
    greater_is_better=True, # Setze auf False, wenn du Loss minimieren willst
    per_device_train_batch_size=16, # Reduziere um OOM zu vermeiden
    per_device_eval_batch_size=16,  # Reduziere um OOM zu vermeiden
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="steps",
    save_strategy= "steps",
    load_best_model_at_end=False,
    logging_steps=750, # Alle XX Schritte loggen
    fp16=True, # Für schnellere und speichereffizientere Berechnung auf GPU
    gradient_checkpointing=True, # Hilft gegen OOM bei großen Modellen, macht Training langsamer
    save_total_limit=1, # Nur das beste Modell speichern
    report_to="none",
    push_to_hub=False, # Zuerst trainieren, dann manuell pushen
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["dev"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics, # Eigene Funktion zur Metrikberechnung
)


In [ ]:
model_name

In [ ]:

# Training
trainer.train()


In [ ]:

print("\nPredictions on dev set after training.")

# Die predict-Methode gibt ein Objekt zurück, das die Vorhersagen,
# die echten Labels und die Metriken enthält (falls der Trainer diese selbst berechnet hat).
predictions_output = trainer.predict(test_dataset=dataset["dev"])

In [ ]:
trainer.push_to_hub()

## Step 2: Train span classifier